<a href="https://colab.research.google.com/github/ahmadfa100/-Ahmad-bani-Hamad/blob/main/Hangging%20face%20vs%20Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install transformers sentencepiece langchain langchain-huggingface --quiet

In [17]:
# Load model directly (pipeline() removed text2text-generation in transformers v5)
import torch, json, warnings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model ready.


In [18]:
# Sample text and fields to extract
text = """
Project Alpha Kickoff Meeting
Date: June 10, 2024
Time: 10:00 AM - 11:30 AM
Location: Conference Room B (also on Zoom)
Attendees: Sarah Chen (PM), James Lopez (Dev Lead), Mia Patel (Designer)
Action Items: James sets up GitHub repo by Wednesday. Mia shares wireframes by Friday.
Priority: High
"""

fields = {
    "event_name":   "What is the name of the event or meeting?",
    "date":         "What is the date mentioned?",
    "time":         "What is the time mentioned?",
    "location":     "What is the location?",
    "attendees":    "Who are the attendees?",
    "action_items": "What are the action items or tasks?",
    "priority":     "What is the priority level?",
}

In [19]:
#  Approach 1: HuggingFace Direct

def hf_generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def hf_extract(input_text):
    results = {}
    for field, question in fields.items():
        prompt = f"{question}\nContext: {input_text.strip()}\nAnswer:"
        results[field] = hf_generate(prompt) or "not found"
    return results

print("=== Approach 1: HuggingFace Direct ===")
hf_result = hf_extract(text)
print(json.dumps(hf_result, indent=2))

=== Approach 1: HuggingFace Direct ===
{
  "event_name": "Project Alpha Kickoff Meeting",
  "date": "June 10, 2024",
  "time": "10:00 AM - 11:30 AM",
  "location": "Conference Room B",
  "attendees": "Sarah Chen (PM), James Lopez (Dev Lead), Mia Patel (Designer)",
  "action_items": "James sets up GitHub repo by Wednesday Mia shares wireframes by Friday",
  "priority": "High"
}


In [20]:
#  Approach 2: LangChain


from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

def run_model(prompt_value):
    # LangChain passes a PromptValue object — convert it to string first
    prompt_str = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    return hf_generate(prompt_str)

prompt_template = PromptTemplate(
    input_variables=["question", "text"],
    template="{question}\nContext: {text}\nAnswer:"
)

chain = prompt_template | RunnableLambda(run_model)

def lc_extract(input_text):
    results = {}
    for field, question in fields.items():
        answer = chain.invoke({"question": question, "text": input_text.strip()})
        results[field] = answer.strip() or "not found"
    return results

print("=== Approach 2: LangChain ===")
lc_result = lc_extract(text)
print(json.dumps(lc_result, indent=2))

# note : Langchain is better if you want to scale your project up, for small projects , use hanging face.

=== Approach 2: LangChain ===
{
  "event_name": "Project Alpha Kickoff Meeting",
  "date": "June 10, 2024",
  "time": "10:00 AM - 11:30 AM",
  "location": "Conference Room B",
  "attendees": "Sarah Chen (PM), James Lopez (Dev Lead), Mia Patel (Designer)",
  "action_items": "James sets up GitHub repo by Wednesday Mia shares wireframes by Friday",
  "priority": "High"
}


In [21]:
#  Side-by-side comparison

print("\n=== Comparison Table ===")
print(f"{'Field':<15} | {'HuggingFace Direct':<38} | {'LangChain'}")
print("-" * 90)
for f in fields:
    hv = hf_result[f]
    lv = lc_result[f]
    tag = "✓" if hv.lower() == lv.lower() else "~"
    print(f"{f:<15} | {hv:<38} | {lv}  {tag}")

print("\n✓ = exact match    ~ = same meaning, different wording")


=== Comparison Table ===
Field           | HuggingFace Direct                     | LangChain
------------------------------------------------------------------------------------------
event_name      | Project Alpha Kickoff Meeting          | Project Alpha Kickoff Meeting  ✓
date            | June 10, 2024                          | June 10, 2024  ✓
time            | 10:00 AM - 11:30 AM                    | 10:00 AM - 11:30 AM  ✓
location        | Conference Room B                      | Conference Room B  ✓
attendees       | Sarah Chen (PM), James Lopez (Dev Lead), Mia Patel (Designer) | Sarah Chen (PM), James Lopez (Dev Lead), Mia Patel (Designer)  ✓
action_items    | James sets up GitHub repo by Wednesday Mia shares wireframes by Friday | James sets up GitHub repo by Wednesday Mia shares wireframes by Friday  ✓
priority        | High                                   | High  ✓

✓ = exact match    ~ = same meaning, different wording


In [22]:
# CELL 7 — Optional: second example to test

text2 = """
Workshop: AI Ethics in Practice
Date: July 18, 2024
Time: 2:00 PM
Location: Room 301, Building C
Speaker: Dr. Omar Al-Farsi
Action Items: Read the pre-workshop paper. RSVP by July 15.
Priority: Medium
"""

print("=== Example 2: HuggingFace Direct ===")
hf2 = hf_extract(text2)
print(json.dumps(hf2, indent=2))

print("\n=== Example 2: LangChain ===")
lc2 = lc_extract(text2)
print(json.dumps(lc2, indent=2))

=== Example 2: HuggingFace Direct ===
{
  "event_name": "Workshop: AI Ethics in Practice",
  "date": "July 18, 2024",
  "time": "2:00 PM",
  "location": "Building C",
  "attendees": "Dr. Omar Al-Farsi",
  "action_items": "Action Items: Read the pre-workshop paper. Priority: Medium",
  "priority": "Medium"
}

=== Example 2: LangChain ===
{
  "event_name": "Workshop: AI Ethics in Practice",
  "date": "July 18, 2024",
  "time": "2:00 PM",
  "location": "Building C",
  "attendees": "Dr. Omar Al-Farsi",
  "action_items": "Action Items: Read the pre-workshop paper. Priority: Medium",
  "priority": "Medium"
}
